## Notebook Instructions

1. If you are new to Jupyter notebooks, please go through this introductory manual [here](https://quantra.quantinsti.com/quantra-notebook).
2. Any changes made in this notebook would be lost after you close the browser window. **You can download the notebook to save your work on your PC.**
3. Before running this notebook on your local PC:
   - You need to set up a Python environment and the relevant packages on your local PC. To do so, go through the section on **"Run Codes Locally on Your Machine"** in the course.
   - You need to **download the zip file available in the last unit** of this course. The zip file contains the data files and/or Python modules that might be required to run this notebook.


# A Complete Automated Trading System

Welcome to the final step of our journey! We've connected to live data, calculated indicators, and generated signals. Now, we will connect our signals directly to Zerodha's order placement system to create an automated, end-to-end trading system.

<font color='red'> <b>IMPORTANT DISCLAIMER:</b>
<BR>The code in this notebook can execute real trades with real money in your trading account. Please be extremely careful. It is highly recommended to test this with a `quantity` of 1 and fully understand the code and its risks before using it in a live environment. You are responsible for all trades placed by this script.</font>

## Outline

1.  **Setup & Initialization**: We will import and initialize both `KiteTicker` (for data) and `KiteConnect` (for orders).
2.  **Order Placement Function**: We'll create a dedicated, clean function to handle placing market orders.
3.  **The Final Code**: We will assemble the complete script, integrating the order function into our `on_ticks` callback.
4.  **Going Live**: We'll run the script that listens, analyzes, and trades automatically.
5.  **Conclusion & Next Steps**: We'll recap our accomplishment and discuss how to build upon this foundation.

### 1. Setup & Initialization

Our setup now requires two main objects:
-   `KiteConnect`: For account interactions like placing and managing orders.
-   `KiteTicker`: For receiving the live WebSocket data stream.

Let's import them and get them ready.

In [1]:
# --- Imports ---
from kiteconnect import KiteConnect, KiteTicker
import logging

# --- Setup ---
logging.basicConfig(level=logging.INFO)

# --- Credentials & Initialization ---
# IMPORTANT: Replace these with your actual api_key and access_token.
# You get the access_token after a successful login via the Kite Connect API.
api_key = "YOUR API KEY"
access_token = "YOUR ACCESS TOKEN"


# 1. Initialize KiteConnect for placing orders
kite = KiteConnect(api_key=api_key)
kite.set_access_token(access_token)
logging.info("KiteConnect (for orders) initialized.")

# 2. Initialize KiteTicker for live data
kws = KiteTicker(api_key, access_token)
logging.info("KiteTicker (for data) initialized.")

INFO:root:KiteConnect (for orders) initialized.
INFO:root:KiteTicker (for data) initialized.


### 2. The Final Code: Integrating Order Execution

This is the complete, final script. We introduce a few key changes:

-   **`place_market_order` function**: A dedicated function to keep our code clean and handle order placement logic, including error handling.
-   **`current_position` variable**: This is our new state-management variable. It will be `None` if we have no open position, and `'LONG'` if we have an open buy position. This prevents the trading system from placing duplicate orders.
-   **Updated `on_ticks` logic**: The core logic now checks for a crossover *and* checks the `current_position` status before calling the `place_market_order` function.
-   **`on_order_update`** function: This function is triggered every time there is an update on any order placed through the trading system. It prints out key details such as the order ID, trading symbol, status, filled quantity, average price, and timestamp. This allows us to monitor the progress and outcome of each order in real-time, making it easier to debug issues or track executions.
While the on_order_update callback provides real-time notifications for order status changes, you can also use the `orders()` or `order_history()` methods to actively fetch the current status of your orders as seen in previous sections.

In [2]:
# --- Strategy & State Variables ---
ltp_history = []
SMA_PERIOD = 10 
instrument_token = 738561      # Example: RELIANCE
trading_symbol = 'RELIANCE'
current_position = None       # Can be 'LONG' or None

# --- Order Placement Function ---
def place_market_order(symbol, transaction_type, quantity=1):
    """Places a market order and logs the response."""
    try:
        order_id = kite.place_order(
            variety=kite.VARIETY_REGULAR,
            exchange=kite.EXCHANGE_NSE,
            tradingsymbol=symbol,
            transaction_type=transaction_type, # Can be kite.TRANSACTION_TYPE_BUY or kite.TRANSACTION_TYPE_SELL
            quantity=quantity,
            product=kite.PRODUCT_MIS,         # Margin Intraday Squareoff
            order_type=kite.ORDER_TYPE_MARKET,
            validity=kite.VALIDITY_DAY
        )
        logging.warning(f"ORDER PLACED: {transaction_type} {quantity} {symbol} | Order ID: {order_id}")
        return order_id
    except Exception as e:
        logging.error(f"Order placement failed for {symbol}: {e}")
        return None

# --- Callback Functions ---
def on_ticks(ws, ticks):
    global current_position
    
    for tick in ticks:
        last_price = tick['last_price']
        ltp_history.append(last_price)
        
        if len(ltp_history) > SMA_PERIOD:
            ltp_history.pop(0)

        if len(ltp_history) == SMA_PERIOD:
            sma = sum(ltp_history) / SMA_PERIOD
            
            # --- STRATEGY & EXECUTION LOGIC ---
            
            # BUY Logic: If price crosses above SMA and we have no position
            if last_price > sma and current_position is None:
                logging.warning(f"BUY SIGNAL: LTP {last_price:.2f} > SMA {sma:.2f}")
                if place_market_order(trading_symbol, kite.TRANSACTION_TYPE_BUY):
                    current_position = 'LONG'

            # SELL Logic: If price crosses below SMA and we are in a long position
            if last_price < sma and current_position == 'LONG':
                logging.warning(f"SELL SIGNAL: LTP {last_price:.2f} < SMA {sma:.2f}")
                if place_market_order(trading_symbol, kite.TRANSACTION_TYPE_SELL):
                    current_position = None

            logging.info(f"LTP: {last_price:.2f} | SMA({SMA_PERIOD}): {sma:.2f} | Position: {current_position}")

def on_connect(ws, response):
    logging.info("WebSocket Connected! Subscribing to instrument token.")
    ws.subscribe([instrument_token])
    ws.set_mode(ws.MODE_LTP, [instrument_token])
    
def on_order_update(ws, order):
    """This is the most important function.
    It's called whenever there is an update to any of your orders."""
    logging.info("--- ORDER UPDATE RECEIVED ---")
    # The 'order' object is a dictionary containing all the order details.
    print(f"Order ID: {order.get('order_id')}")
    print(f"Tradingsymbol: {order.get('tradingsymbol')}")
    print(f"Status: {order.get('status')}")
    print(f"Filled Quantity: {order.get('filled_quantity')}")
    print(f"Average Price: {order.get('average_price')}")
    print(f"Order Timestamp: {order.get('order_timestamp')}")
    print("-" * 30)

def on_close(ws, code, reason):
    logging.error(f"WebSocket connection closed: {code} - {reason}")
    ws.stop()

### 3. Going Live

This is the final step. The cell below will assign the callbacks and start the connection. Once you run it, the script will be live. It will monitor the price of the instrument, and if the crossover conditions are met, <font color='red'>**it will place a real trade in your account.**</font>

In [3]:
# --- Assign Callbacks and Connect ---
kws.on_ticks = on_ticks
kws.on_connect = on_connect
kws.on_close = on_close
kws.on_order_update = on_order_update

print("Connecting to WebSocket... (This will run indefinitely. Interrupt the kernel to stop.)")
print("---REMINDER: THIS SCRIPT WILL PLACE LIVE TRADES---")
kws.connect()

Connecting to WebSocket... (This will run indefinitely. Interrupt the kernel to stop.)
---REMINDER: THIS SCRIPT WILL PLACE LIVE TRADES---


INFO:root:WebSocket Connected! Subscribing to instrument token.
INFO:root:LTP: 1377.60 | SMA(10): 1377.43 | Position: LONG
INFO:root:--- ORDER UPDATE RECEIVED ---
INFO:root:--- ORDER UPDATE RECEIVED ---


Order ID: 250908100744928
Tradingsymbol: RELIANCE
Status: OPEN
Filled Quantity: 0
Average Price: 0
Order Timestamp: 2025-09-08 12:30:57
------------------------------
Order ID: 250908100744928
Tradingsymbol: RELIANCE
Status: COMPLETE
Filled Quantity: 1
Average Price: 1377.7
Order Timestamp: 2025-09-08 12:30:57
------------------------------


INFO:root:LTP: 1377.70 | SMA(10): 1377.45 | Position: LONG
INFO:root:LTP: 1377.70 | SMA(10): 1377.48 | Position: LONG
INFO:root:LTP: 1377.70 | SMA(10): 1377.51 | Position: LONG
INFO:root:LTP: 1377.70 | SMA(10): 1377.54 | Position: LONG
INFO:root:LTP: 1377.70 | SMA(10): 1377.57 | Position: LONG
INFO:root:LTP: 1377.70 | SMA(10): 1377.60 | Position: LONG
INFO:root:LTP: 1377.70 | SMA(10): 1377.63 | Position: LONG
INFO:root:LTP: 1377.70 | SMA(10): 1377.66 | Position: LONG
INFO:root:LTP: 1377.70 | SMA(10): 1377.69 | Position: LONG
INFO:root:LTP: 1377.40 | SMA(10): 1377.67 | Position: None
INFO:root:LTP: 1377.40 | SMA(10): 1377.64 | Position: None
INFO:root:--- ORDER UPDATE RECEIVED ---
INFO:root:--- ORDER UPDATE RECEIVED ---


Order ID: 250908100745132
Tradingsymbol: RELIANCE
Status: OPEN
Filled Quantity: 0
Average Price: 0
Order Timestamp: 2025-09-08 12:31:04
------------------------------
Order ID: 250908100745132
Tradingsymbol: RELIANCE
Status: COMPLETE
Filled Quantity: 1
Average Price: 1377.7
Order Timestamp: 2025-09-08 12:31:04
------------------------------


INFO:root:LTP: 1377.70 | SMA(10): 1377.64 | Position: LONG
INFO:root:LTP: 1377.70 | SMA(10): 1377.64 | Position: LONG
INFO:root:--- ORDER UPDATE RECEIVED ---
INFO:root:--- ORDER UPDATE RECEIVED ---


Order ID: 250908100745162
Tradingsymbol: RELIANCE
Status: OPEN
Filled Quantity: 0
Average Price: 0
Order Timestamp: 2025-09-08 12:31:05
------------------------------
Order ID: 250908100745162
Tradingsymbol: RELIANCE
Status: COMPLETE
Filled Quantity: 1
Average Price: 1377.8
Order Timestamp: 2025-09-08 12:31:05
------------------------------


INFO:root:LTP: 1377.80 | SMA(10): 1377.65 | Position: LONG
INFO:root:LTP: 1377.80 | SMA(10): 1377.66 | Position: LONG
ERROR:kiteconnect.ticker:Connection error: 1006 - connection was closed uncleanly (peer dropped the TCP connection without previous WebSocket closing handshake)
ERROR:kiteconnect.ticker:Connection closed: 1006 - connection was closed uncleanly (peer dropped the TCP connection without previous WebSocket closing handshake)
ERROR:root:WebSocket connection closed: 1006 - connection was closed uncleanly (peer dropped the TCP connection without previous WebSocket closing handshake)
Unhandled Error
Traceback (most recent call last):
  File "/home/nb-user/.conda/envs/quantra_py/lib/python3.11/site-packages/twisted/python/log.py", line 96, in callWithLogger
    return callWithContext({"system": lp}, func, *args, **kw)
  File "/home/nb-user/.conda/envs/quantra_py/lib/python3.11/site-packages/twisted/python/log.py", line 80, in callWithContext
    return context.call({ILogContext:

### 4. Adding a Margin Check to the System

You can also include a margin check to make sure you have the required capital for placing orders. For this exercise, we will create a hypothetical order of Reliance.

In [2]:
# Define the order parameters for which you want to calculate the margin
# Example: Buying 100 shares of Reliance (RELIANCE) for MIS Intraday

order_params_single = [
    {
        "exchange": "NSE",
        "tradingsymbol": "RELIANCE",
        "transaction_type": "BUY",
        "variety": "regular",
        "product": "MIS",        # Margin Intraday Square Off
        "order_type": "LIMIT",
        "quantity": 100,
        "price": 2800.00,        # Price for LIMIT order
        "trigger_price": 0.0     # Not used for LIMIT/MARKET
    }
]

print("Order to check margin for: RELIANCE (BUY, 100 Qty, MIS)")

Order to check margin for: RELIANCE (BUY, 100 Qty, MIS)


You will now check the margin requirement for this order using the `kite.order_margins` method.

In [3]:
def check_single_order_margin(order_params):
    """
    Calculates the margin required for a single, hypothetical order 
    using the syntax for kiteconnect.
    """
    print("\n--- Margin Calculation Result (Full Detail) ---")
    try:
        # 1. Pass the list of orders directly
        margin_response = kite.order_margins(order_params)
                
        # The total required margin is usually found under the 'total' key 
        # inside the first element of the response list.
        required_margin = margin_response[0]['total']
        
        print(f"Total Margin Required: ₹ {required_margin:,.2f}")
        print("-------------------------------------------------")
        
    except Exception as e:
        print(f"Failed to calculate margin with old syntax: {e}")

# Call the function with your order parameters
check_single_order_margin(order_params_single)


--- Margin Calculation Result (Full Detail) ---
Total Margin Required: ₹ 27,500.00
-------------------------------------------------


You can also check the margin requirement for spread orders as well. The Margin calculation APIs lets you calculate `span`, `exposure`, `option premium`, `additional`, `bo`, `cash`, `var`, `pnl values` for a list of orders as well. For more information, check the documentation here: <a href="https://kite.trade/docs/connect/v3/margins/" target="_blank" rel="noopener">Margin calculation</a>.

You can use the postback (webhook) method to get arbitrary updates to your orders reliably, irrespective of when they happen (COMPLETE, CANCEL, REJECTED, UPDATE). An UPDATE postback is triggered when an open order is modified or when there's a partial fill. This can be used to track trades. For more information, you can check the documentation at the following link: <a href="https://kite.trade/docs/connect/v3/postbacks/" target="_blank" rel="noopener">Postback (WebHooks)</a>


### 5. Conclusion & Where to Go From Here

Congratulations! You have officially built and deployed a complete, end-to-end automated trading system. It receives data, analyzes it, and acts on its logic without any human intervention. You also know how to check margin requirement for large orders.

This is a massive achievement, but it's also a foundational starting point. The world of algorithmic trading is vast. Here are some ideas for where to go next:

-   **Risk Management**: Implement stop-loss and take-profit orders.
-   **Position Sizing**: Instead of trading a fixed quantity of 1, calculate the quantity based on your risk tolerance and account size.
-   **More Complex Indicators**: Combine the SMA with other indicators like RSI or MACD for more robust signals.
-   **Robust Error Handling**: What if the WebSocket disconnects? Add logic in `on_close` to try and reconnect automatically.
-   **Backtesting**: Before deploying a new strategy with real money, always backtest it on historical data to see how it would have performed.<br><br>